# Project K, Series Two: MiniCPM-o 2.6 diagnostic run

**Why this notebook exists.** Series One showed that MiniCPM-o 2.6
reported water and missed most other sounds. The council then asked a
harder question: which stage loses the sound? The feed audit (Series
Two test 1) proved the old runner had a silent failure path: a missing
track could be padded with zeros and leave no trace. This notebook is
built so that cannot happen again.

**What this notebook does.** It runs three diagnostic tests on the
same MiniCPM-o 2.6, through the same door as the Series One run:

1. **Test 2, the 2x2.** Four sounds (speech, true silence, water,
   beeps), each fed as sound alone and as sound with the swap still
   picture, crossed with an open question and direct questions.
2. **Test 3, yes/no scoring.** The full seven-file swap set, in three
   versions each: sound on with the still picture, muted with the
   still picture, and sound on with a plain gray frame.
3. **Test 4, tone-shift timing.** One 1000 Hz tone, 0.5 seconds long,
   placed at about 2, 6, and 10 seconds in 12 second files. Presence,
   early-or-late, and the numeric start are asked separately.

**The feed proof rule.** Before every answer is taken, the runner
prints the proof for that exact feed: both ffmpeg return codes, the
extracted track duration, sample rate, sample count, overall RMS in
dB, nonzero share, the unit count, every one-second unit's RMS, the
zero-chunk count, any padding samples, and a hash of the exact decoded
track handed toward the model. A copy of each unique extracted track
is also saved beside the results. If a track that should carry sound
decodes missing, empty, short, or all zeros, the run hard-stops with
a loud banner. Declared silence and declared muted versions are
exempt by name in the job list, never by accident. Nothing is padded
quietly: a short final chunk is padded only to complete its second,
and the padded sample count is printed in that ask's proof.

**The yes-score fact, settled from the code before any run.**
MiniCPM's `model.chat` does not expose a yes-token score. Reading
`modeling_minicpmo.py`: `chat` calls `generate`, `_decode` asks the
language model for sequences and hidden states but never requests
output scores, and `chat` returns only decoded text (or an
`OmniOutput` holding text). So every row's yes-score column reads
`not exposed`, and the plain yes or no words are the record. No
internal path is substituted, because that would no longer be the
chat door being tested.

**How to run it.** Run the cells from the top, in order. Step 1
builds the same private environment as the MiniCPM Series One
notebook and proves the stack. Step 2 gets the seven swap files,
builds the gray-frame versions and the three tone files, measures
them from their decoded tracks, and saves the new files into the
Drive clips folder so OLA can ride the identical cells later. Step 3
writes the runner and runs its check mode (proof, loading ladder,
sanity). Step 4 runs the diagnostic battery and saves
`series2_minicpm_results.csv` in the Drive clips folder.

## Step 1 · Setup: a private environment for the model's stack

The next cell does four things:

1. Installs `uv`, a fast environment and package tool, if this machine
   does not already have it. That is the only thing it adds to Colab's
   own Python, and it is a tool, not part of the model's stack.
2. Creates a private virtual environment at `/content/minicpm-venv`,
   on Python 3.12, the same Python generation Colab itself runs. The
   model's stack was proven to install cleanly on Python 3.12, so the
   private environment keeps 3.12 and changes only who owns it.
3. Installs the model's stack inside that environment, in two passes.
   First, torch and torchaudio pinned to 2.8.0 from PyTorch's CUDA 12.6
   shelf: that is the exact build this runtime already runs, so the
   chip driver match is the proven one, and the installer cannot drag
   in a multi-gigabyte CUDA 13 torch the driver may not support (left
   floating, the resolver does exactly that). Second, the model card's
   own pin list, unchanged: transformers 4.44.2, accelerate 0.34.2,
   bitsandbytes 0.46.1, librosa 0.10.2.post1 (a newer patch than the
   card lists, because the card's version predates Colab's current
   numpy), soundfile, vocos, and vector-quantize-pytorch. Pillow rides
   along because the model's code and the battery both need it and
   nothing in the pin list pulls it in. vocos and
   vector-quantize-pytorch are parts the model's code imports for its
   speech side; they are installed even though this test never asks
   the model to speak. One more install rides after those two passes:
   flash-attn 2.8.3, the attention library the model's own modeling
   file imports. Transformers checks that file's imports before it
   loads a single weight, so without flash-attn the model cannot
   even start. Colab's stock environment carries it already, which
   is why the model card never lists it; the private environment
   gets its own copy as the official prebuilt wheel for this exact
   stack (Python 3.12, torch 2.8, CUDA 12, cxx11 ABI), pinned by its
   full download address, never a build from source.
4. Proves the stack, in a fresh process under the environment's own
   Python. The proof prints the Python version, the torch version, the
   transformers version and the file it loads from, and it imports
   `WHISPER_ATTENTION_CLASSES` from the transformers whisper module,
   the exact symbol the model's own code asks for and newer
   transformers versions removed, and it imports flash_attn and
   prints its version. A version number alone is not proof anymore;
   the imports are. If any line of the proof is wrong, Step 1 stops
   on the spot and the run goes no further.

Nothing in this cell installs over, downgrades, or removes anything
Colab shipped. The two failed runs of this notebook both died in that
fight; this design simply refuses to have it.

The install downloads a few gigabytes (torch is the big one) and takes
several minutes. That is normal. Every command is printed before it
runs, and the installers stream their output into the cell, so nothing
about this step happens off the record.

In [ ]:
# --- Setup: build the private environment and prove the stack inside it --------
# This cell never installs the model's stack into Colab's Python. Two
# earlier runs of this notebook died doing exactly that: the pinned
# transformers never landed where the session reads. The stack now
# lives in a private virtual environment at /content/minicpm-venv, and
# Step 1 ends by proving, in a fresh process under that environment's
# Python, that the pinned versions and the one symbol the model's code
# needs are really there.
import os, sys, shutil, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

VENV = "/content/minicpm-venv"
VENV_PY = os.path.join(VENV, "bin", "python")

def run_stream(cmd):
    print("+", " ".join(cmd))
    proc = subprocess.run(cmd)
    print("  exit code:", proc.returncode)
    return proc.returncode

# 1. uv: the environment tool. The only addition to Colab's own Python.
if shutil.which("uv") is None:
    print("uv not found. Installing it (a tool only, not the model stack).")
    rc = run_stream([sys.executable, "-m", "pip", "install", "uv"])
    if rc != 0 or shutil.which("uv") is None:
        print()
        print("!" * 72)
        print("STOP. uv could not be installed, so the private environment")
        print("cannot be built. Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: uv install failed")
print("uv:", shutil.which("uv"))

# 2. The private environment itself, on Python 3.12. The card's stack
# was proven to install cleanly on Python 3.12 (a full clean resolution
# of the exact pin list), so the private environment keeps 3.12 and
# changes only who owns the environment: this run, not Colab.
if not os.path.exists(VENV_PY):
    rc = run_stream(["uv", "venv", "--python", "3.12", VENV])
    if rc != 0:
        print()
        print("!" * 72)
        print("STOP. The private environment could not be created.")
        print("Show this output to Muse.")
        print("!" * 72)
        raise SystemExit("STOP: venv creation failed")
ver = subprocess.run([VENV_PY, "--version"], capture_output=True, text=True)
print("venv python:", (ver.stdout or ver.stderr).strip())

# 3a. torch first, pinned to the runtime's own proven build. Colab's
# pip list on the failed runs showed torch 2.8.0+cu126: that is the
# build this chip and driver already run. Left floating, the resolver
# picks a newest torch built on CUDA 13 and downloads gigabytes of
# driver libraries the runtime may not support. Pinning it is not a
# change to the model card's list; the card leaves torch to the
# machine, and this makes the machine's proven choice explicit.
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 "torch==2.8.0", "torchaudio==2.8.0",
                 "--index-url", "https://download.pytorch.org/whl/cu126"])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The torch install inside the private environment failed.")
    print("The installer's own output is above. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: torch install failed in venv")

# 3b. The model card's pin list, unchanged, in one resolver pass. No
# force-reinstall: the environment is empty and private, so there is
# nothing to force against. Pillow is added because the model's code
# and the battery import it and no package in the list pulls it in.
CARD_PACKAGES = [
    "transformers==4.44.2",
    "accelerate==0.34.2",
    "bitsandbytes==0.46.1",
    "librosa==0.10.2.post1",
    "soundfile==0.13.1",
    "vocos==0.1.0",
    "vector-quantize-pytorch==1.18.5",
    "Pillow",
]
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY] + CARD_PACKAGES)
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The model card's stack did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: card stack install failed in venv")

# 3c. flash-attn: the one package the sealed environment was missing.
# MiniCPM's own modeling file imports flash_attn, and transformers
# checks that file's imports before it loads a single weight. The
# fifth round's full-capture log (minicpm_load_log.txt) shows all
# three rungs dying in about three seconds on exactly that check:
# "This modeling file requires the following packages that were not
# found in your environment: flash_attn". Colab's stock environment
# carries flash-attn 2.8.3 already, which is why the model card never
# lists it; the private environment has to be given its own copy. It
# goes in as the official prebuilt wheel, never a build from source:
# Dao-AILab's flash-attn 2.8.3 for CPython 3.12, torch 2.8, and CUDA
# 12, in the cxx11 ABI TRUE form, which is the ABI torch 2.8 itself
# ships with (PyTorch dropped its pre-cxx11 ABI builds at 2.8). The
# address was fetched and the wheel opened when this round was
# built: it answers HTTP 200, it holds the cp312 CUDA extension, and
# its version reads 2.8.3. Its one extra dependency, einops, is
# small and comes from the normal shelf in the same pass.
FLASH_ATTN_WHEEL = ("https://github.com/Dao-AILab/flash-attention/"
                    "releases/download/v2.8.3/flash_attn-2.8.3"
                    "%2Bcu12torch2.8cxx11abiTRUE-cp312-cp312"
                    "-linux_x86_64.whl")
rc = run_stream(["uv", "pip", "install", "--python", VENV_PY,
                 FLASH_ATTN_WHEEL])
if rc != 0:
    print()
    print("!" * 72)
    print("STOP. The flash-attn wheel did not install inside the private")
    print("environment. The installer's own output is above, complete.")
    print("Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: flash-attn install failed in venv")

# 4. Proof, in a fresh process under the environment's Python. Version
# alone is not accepted as proof anymore: the proof also imports
# WHISPER_ATTENTION_CLASSES, the exact symbol MiniCPM's own code asks
# transformers for, whose absence is what a wrong transformers version
# actually breaks, and flash_attn, the package the model's modeling
# file demands at the door (see 3c above).
PROOF = (
    "import sys, torch, transformers\n"
    "print('venv python:', sys.version.split()[0])\n"
    "print('torch version:', torch.__version__)\n"
    "print('transformers version:', transformers.__version__)\n"
    "print('transformers file:', transformers.__file__)\n"
    "from transformers.models.whisper.modeling_whisper import "
    "WHISPER_ATTENTION_CLASSES\n"
    "print('WHISPER_ATTENTION_CLASSES import: OK', "
    "sorted(WHISPER_ATTENTION_CLASSES.keys()))\n"
    "import flash_attn\n"
    "print('flash_attn version:', flash_attn.__version__)\n"
)
proof = subprocess.run([VENV_PY, "-c", PROOF],
                       capture_output=True, text=True)
print()
print(proof.stdout)
if proof.stderr.strip():
    print("proof process also said:")
    print("\n".join(proof.stderr.splitlines()[-10:]))
ok = ("transformers version: 4.44.2" in proof.stdout
      and "WHISPER_ATTENTION_CLASSES import: OK" in proof.stdout
      and "flash_attn version: 2.8.3" in proof.stdout)
if not ok:
    print("!" * 72)
    print("STOP. THE PROOF FAILED INSIDE THE PRIVATE ENVIRONMENT.")
    print("Either a version is wrong (transformers 4.44.2, flash_attn")
    print("2.8.3) or one of the proof imports failed: the whisper symbol")
    print("or flash_attn. Do not go on to Step 2. Show this to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: venv proof failed")
print("proof passed: transformers 4.44.2, loaded from the path above,")
print("WHISPER_ATTENTION_CLASSES imports from it, and flash_attn")
print("2.8.3 imports too.")

# The chip, read from the machine itself. This cell deliberately never
# imports torch or transformers into the notebook's own session.
smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print()
print((smi.stdout or smi.stderr).strip())
print("This notebook wants an L4 (about 24 GB). A T4 (about 16 GB)")
print("cannot hold the model in full form; the ladder would have to")
print("squeeze it, and the record would say so.")
print("Step 1 done. Go on to Step 2. No restart is needed.")

## Step 2 · Get the swap files and build the new versions

The seven swap files are already in the Drive folder named
`project-k-clips`: `swap_silence.mp4`, `swap_speech.mp4`,
`swap_beeps.mp4`, `swap_water.mp4`, `swap_vacuum.mp4`,
`swap_sizzle.mp4`, and `swap_mixed.mp4`. One still picture, seven
soundtracks, each 10 seconds, loudness matched at build time to
about RMS -20 dB (silence is true digital silence).

This cell also builds two new families of files, measures each one
from its decoded track, and saves them into the same Drive clips
folder so a later model (OLA first) can ride identical cells:

1. **Gray-frame versions** (`series2_gray_<sound>.mp4`): each swap
   soundtrack over a plain gray frame, same size and length as the
   swap still versions.
2. **Tone files** (`series2_tone_2s.mp4`, `series2_tone_6s.mp4`,
   `series2_tone_10s.mp4`): the swap still picture for 12 seconds,
   carrying one 1000 Hz tone, 0.5 seconds long, the same single tone
   the probe file uses, placed at about 2, 6, and 10 seconds. Each
   file's total track level is matched to RMS -20 dB by a measured
   gain loop, then the finished MP4 is decoded back and its tone
   placement is measured and printed.

Do not go on until all seven swap files say ready and the build
table prints a measured placement for each tone file.

In [ ]:
# --- Get the swap files; build and measure the gray and tone versions ----------
import os, shutil, subprocess, json, math, wave, hashlib
import numpy as np

SWAP = ["silence", "speech", "beeps", "water", "vacuum", "sizzle", "mixed"]
WORK = "/content/clips"
DRIVE_CLIPS = "/content/drive/MyDrive/project-k-clips"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/project-k-clips"]

for sound in SWAP:
    name = f"swap_{sound}.mp4"
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        continue
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            break

missing = [s for s in SWAP
           if not os.path.exists(os.path.join(WORK, f"swap_{s}.mp4"))]
if missing:
    print("MISSING swap files:", missing)
    raise SystemExit("STOP: swap files missing; do not go on")

def ffprobe_duration(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path], capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

def decode_mono(path, sr=48000):
    proc = subprocess.run(
        ["ffmpeg", "-v", "error", "-i", path, "-vn", "-ac", "1",
         "-ar", str(sr), "-f", "f32le", "-"],
        capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(f"decode failed for {path}: "
                           + proc.stderr.decode(errors="replace")[-500:])
    return np.frombuffer(proc.stdout, dtype=np.float32), sr

def rms_db(x):
    if len(x) == 0:
        return -120.0
    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))
    return 20.0 * math.log10(v) if v > 0 else -120.0

def save_both(local_path):
    # Keep the work copy and, when Drive is mounted, the clips-folder
    # copy that later models will ride.
    if os.path.isdir(DRIVE_CLIPS):
        shutil.copy(local_path, os.path.join(DRIVE_CLIPS,
                                             os.path.basename(local_path)))
        print("saved to Drive clips folder:", os.path.basename(local_path))

build_log = []

# ---- gray-frame versions -------------------------------------------------------
for sound in SWAP:
    out = os.path.join(WORK, f"series2_gray_{sound}.mp4")
    if not os.path.exists(out):
        src = os.path.join(WORK, f"swap_{sound}.mp4")
        proc = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-f", "lavfi",
             "-i", "color=c=0x808080:s=854x480:r=25:d=10",
             "-i", src, "-map", "0:v", "-map", "1:a", "-t", "10",
             "-c:v", "libx264", "-pix_fmt", "yuv420p",
             "-c:a", "aac", "-ar", "48000", "-shortest", out])
        if proc.returncode != 0 or not os.path.exists(out):
            raise SystemExit(f"STOP: gray build failed for {sound}")
    x, sr = decode_mono(out)
    entry = {"file": os.path.basename(out),
             "duration_s": round(len(x) / sr, 3),
             "audio_rms_db_decoded": round(rms_db(x), 2),
             "nonzero_share": round(float(np.count_nonzero(x) / len(x)), 4)}
    build_log.append(entry)
    save_both(out)
    print("gray:", entry)

# ---- tone files ----------------------------------------------------------------
still = os.path.join(WORK, "series2_still.png")
subprocess.run(["ffmpeg", "-y", "-v", "error",
                "-i", os.path.join(WORK, "swap_speech.mp4"),
                "-frames:v", "1", still], check=True)

TONE_SR = 48000
TONE_DUR = 12.0
TARGET_RMS = -20.0

def write_tone_wav(path, start_s, amplitude):
    n = int(TONE_SR * TONE_DUR)
    x = np.zeros(n, dtype=np.float64)
    m = int(TONE_SR * 0.5)
    s = int(round(start_s * TONE_SR))
    t = np.arange(m) / TONE_SR
    x[s:s + m] = amplitude * np.sin(2.0 * np.pi * 1000.0 * t)
    pcm = np.clip(np.round(x * 32767), -32768, 32767).astype("<i2")
    with wave.open(path, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(TONE_SR)
        w.writeframes(pcm.tobytes())

def measure_tone(path):
    x, sr = decode_mono(path, TONE_SR)
    peak = float(np.max(np.abs(x))) if len(x) else 0.0
    thr = max(0.03, peak * 0.05)
    idx = np.where(np.abs(x) > thr)[0]
    onset = float(idx[0] / sr) if len(idx) else None
    return {"duration_s": round(len(x) / sr, 3),
            "audio_rms_db_decoded": round(rms_db(x), 2),
            "peak": round(peak, 3),
            "measured_onset_s": None if onset is None else round(onset, 3),
            "nonzero_share": round(float(np.count_nonzero(x) / len(x)), 4)}

for start_s in (2, 6, 10):
    out = os.path.join(WORK, f"series2_tone_{start_s}s.mp4")
    wav_path = os.path.join(WORK, f"series2_tone_{start_s}s.wav")
    amplitude = 0.6928
    entry = None
    for attempt in range(6):
        write_tone_wav(wav_path, start_s, amplitude)
        proc = subprocess.run(
            ["ffmpeg", "-y", "-v", "error", "-loop", "1", "-framerate", "25",
             "-i", still, "-i", wav_path, "-t", "12",
             "-c:v", "libx264", "-pix_fmt", "yuv420p", "-r", "25",
             "-c:a", "aac", "-ar", str(TONE_SR), "-shortest", out])
        if proc.returncode != 0 or not os.path.exists(out):
            raise SystemExit(f"STOP: tone build failed for {start_s}s")
        entry = measure_tone(out)
        if abs(entry["audio_rms_db_decoded"] - TARGET_RMS) <= 0.7:
            break
        amplitude *= 10 ** ((TARGET_RMS - entry["audio_rms_db_decoded"]) / 20.0)
        if amplitude >= 0.98:
            raise SystemExit(f"STOP: tone gain loop ran away for {start_s}s")
    entry = {"file": os.path.basename(out), "target_start_s": start_s,
             **entry, "amplitude": round(float(amplitude), 4)}
    build_log.append(entry)
    save_both(out)
    print("tone:", entry)

log_path = os.path.join(WORK, "series2_build_log.json")
with open(log_path, "w") as fh:
    json.dump(build_log, fh, indent=2)
if os.path.isdir(DRIVE_CLIPS):
    shutil.copy(log_path, os.path.join(DRIVE_CLIPS,
                                       "series2_build_log.json"))
print()
print("ready: all 7 swap files, 7 gray versions, 3 tone files")
print("build log:", log_path)

## Step 3 · Write the runner, then check: ladder and sanity

The first cell below writes the runner script,
`/content/minicpm_series2_run.py`. It holds the same proof, the same
loading ladder (bf16, then 8-bit, then 4-bit, all pinned to revision
`06849bfd36da94da1f5a88fa17e8ba63f08a4c4d`, remote-code cache cleared
first), and the same sanity gate as the Series One MiniCPM runner.
What changed is only what a failure leaves behind and what the
battery asks: rung tracebacks now append to `series2_load_log.txt`
in the clips folder, and the battery is the Series Two diagnostic
set, not the twenty Series One questions.

The second cell runs the script in **check mode**: proof lines (now
including the yes-score fact: `model.chat` exposes no score, settled
from the code), the ladder, and the sanity question. If any of those
fail, the script stops and Step 4 must not be run.

In [ ]:
# --- Write the Series Two runner script ----------------------------------------
# The runner holds the ladder, the sanity check, the strict feed
# proof, and the diagnostic battery. It is written to a plain file
# here and only ever run under the private environment's Python.
RUNNER_SOURCE = '\n# minicpm_series2_run.py - the Series Two diagnostic runner for MiniCPM-o 2.6.\n# Written by project_k_minicpm_series2_colab.ipynb. Run only under the\n# private environment\'s Python (/content/minicpm-venv/bin/python).\n# Modes: "check" runs the proof, the loading ladder, and the sanity\n# question, then stops. "battery" does all of that and then runs the\n# Series Two diagnostic tests and saves the CSV.\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nfrom transformers.models.whisper.modeling_whisper import WHISPER_ATTENTION_CLASSES\nprint("WHISPER_ATTENTION_CLASSES import: OK",\n      sorted(WHISPER_ATTENTION_CLASSES.keys()))\nif transformers.__version__ != "4.44.2":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.44.2.")\n    sys.exit(3)\n\n# The yes-score question, settled from the model\'s own code before the\n# run (modeling_minicpmo.py in the pinned revision): MiniCPMO.chat\n# calls generate, _decode requests sequences and hidden states from\n# llm.generate but never output scores, and chat returns only the\n# decoded text. There is no score surface on the chat door, so this\n# constant is False and every CSV row marks the score as not exposed.\nYES_SCORE_EXPOSED = False\nprint("yes token score exposed by model.chat:", YES_SCORE_EXPOSED,\n      "(chat returns decoded text only; output scores are never requested)")\n\n# --- Load the model: the same MiniCPM-o ladder as Series One --------------------\nimport traceback, shutil, datetime\nfrom transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig\n\nREPO = "openbmb/MiniCPM-o-2_6"\nREVISION = "06849bfd36da94da1f5a88fa17e8ba63f08a4c4d"\n\nLOAD_LOG_PATHS = ["/content/drive/MyDrive/project-k-clips/series2_load_log.txt",\n                  "/content/series2_load_log.txt"]\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("MiniCPM-o 2.6 Series Two load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\nlog_load("repo: " + REPO + " at pinned revision " + REVISION + "\\n")\n\ntry:\n    from transformers.dynamic_module_utils import HF_MODULES_CACHE\n    module_cache = os.path.join(str(HF_MODULES_CACHE), *REPO.split("/"))\n    if os.path.isdir(module_cache):\n        shutil.rmtree(module_cache)\n        clear_note = "cleared cached remote code for this repo: " + module_cache\n    else:\n        clear_note = ("no cached remote code for this repo to clear: "\n                      + module_cache)\nexcept Exception as e:\n    clear_note = "WARNING: the remote-code cache clear did not run: " + str(e)\nprint(clear_note)\nlog_load(clear_note + "\\n")\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\ntokenizer = None\nMODEL_NAME = None\nLOAD_MODE = None\n\ndef try_load(mode):\n    kwargs = dict(trust_remote_code=True, attn_implementation="sdpa",\n                  init_vision=True, init_audio=True, init_tts=False)\n    if mode == "bf16":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16, **kwargs)\n        m = m.eval().cuda()\n    elif mode == "8-bit":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16, load_in_8bit=True,\n            device_map="cuda", **kwargs)\n        m = m.eval()\n    elif mode == "4-bit":\n        m = AutoModel.from_pretrained(\n            REPO, revision=REVISION, torch_dtype=torch.bfloat16,\n            quantization_config=BitsAndBytesConfig(\n                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16),\n            device_map="cuda", **kwargs)\n        m = m.eval()\n    t = AutoTokenizer.from_pretrained(REPO, revision=REVISION,\n                                      trust_remote_code=True)\n    return m, t\n\nrung_errors = []\nfor mode in ("bf16", "8-bit", "4-bit"):\n    print("=" * 70)\n    print(f"rung: {REPO} in {mode} form")\n    try:\n        m, t = try_load(mode)\n        model, tokenizer = m, t\n        MODEL_NAME, LOAD_MODE = REPO, mode\n        print("loaded:", REPO, "in", mode, "form")\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        rung_errors.append((REPO, mode, tb))\n        print("this rung failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("rung failed: " + REPO + " in " + mode + " form\\n")\n        log_load(tb)\n        model = None\n        tokenizer = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("ALL THREE RUNGS FAILED. MiniCPM-o 2.6 did not load in any fit.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every rung is saved, word for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\n# --- Sanity check: can the model write at all? ---------------------------------\ndef answer_text(res):\n    if isinstance(res, dict):\n        return str(res.get("text", "") or "")\n    if isinstance(res, (list, tuple)):\n        return answer_text(res[0]) if res else ""\n    return str(res or "")\n\ntry:\n    msgs = [{"role": "user",\n             "content": ["Reply with the word ready."]}]\n    sres = model.chat(msgs=msgs, tokenizer=tokenizer,\n                      sampling=False, max_new_tokens=16)\n    sraw = answer_text(sres).strip()\n    print("SANITY RAW OUTPUT:", repr(sraw))\n    if sraw:\n        print("sanity check passed: the model can write.")\n    else:\n        print("WARNING: the sanity check came back EMPTY.")\n        print("STOP. Show this output to Muse.")\n        sys.exit(5)\nexcept SystemExit:\n    raise\nexcept Exception as e:\n    print("WARNING: the sanity check FAILED with an error:")\n    print(str(e))\n    print("STOP. Show this output to Muse.")\n    sys.exit(5)\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, ladder, and sanity all passed.")\n    print("The diagnostic battery was not run. Run Step 4 for the tests.")\n    sys.exit(0)\n\n# --- Diagnostic battery ---------------------------------------------------------\nimport csv, glob, hashlib, json, math, re, subprocess\nimport numpy as np\nimport librosa\nimport soundfile as sf\nfrom PIL import Image\n\n# The repaired JSON question, injected at build time from\n# pilot2/run_eggs.py, character for character (444 characters).\nPROMPT_JSON = \'You are given a short first-person video clip. Watch and listen to the whole clip.\\nRespond with ONLY a JSON object in exactly this form, no other text: {"seen": "<the main things happening that you can SEE, short>", "heard": "<the main things happening that you can HEAR, short; write the word none if you hear no sound at all>", "sound_start_s": <number: the second at which the main sound event starts, or the word null if there is no sound>}\'\nassert len(PROMPT_JSON) == 444, f"prompt drifted: {len(PROMPT_JSON)} chars"\n\nCLIPS = "/content/clips"\nTRACK_DIRS = ["/content/drive/MyDrive/project-k-clips/series2_extracted_tracks",\n              "/content/series2_extracted_tracks"]\nfor d in TRACK_DIRS:\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\nFEED_WITH_PICTURE = ("omni chat units: 1 frame per second + the matching 1s "\n                     "of the 16 kHz audio track per unit, frames downscaled "\n                     "to fit 448x448, max 15 units, max_slice_nums=1, "\n                     "greedy (sampling=False), no generated speech")\nFEED_AUDIO_ALONE = ("closest audio-only unified chat form: model.chat "\n                    "accepts numpy audio with no image; the code has no "\n                    "separate audio-only method, so the unified chat is "\n                    "used with the default audio-and-text system prompt, "\n                    "per-second 16 kHz audio chunks, no picture and no "\n                    "unit markers, greedy (sampling=False), "\n                    "no generated speech")\n\nclass FeedProofError(RuntimeError):\n    pass\n\ndef rms_db(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef media_duration(path):\n    out = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "csv=p=0", path], capture_output=True, text=True)\n    try:\n        return float(out.stdout.strip())\n    except Exception:\n        return None\n\ndef file_sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for block in iter(lambda: fh.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\nFRAME_DIR = "/content/series2_frames"\nos.makedirs(FRAME_DIR, exist_ok=True)\n_feed_cache = {}\n\ndef extract_feed(path, with_picture, expected_sound, expected_silence):\n    key = (path, bool(with_picture), bool(expected_sound),\n           bool(expected_silence))\n    if key in _feed_cache:\n        return _feed_cache[key]\n    if not os.path.exists(path):\n        raise FeedProofError(f"feed file missing: {path}")\n    tag = re.sub(r"[^A-Za-z0-9]+", "_",\n                 os.path.basename(path) + "_" + str(with_picture))\n    fdir = os.path.join(FRAME_DIR, tag)\n    os.makedirs(fdir, exist_ok=True)\n    for old in glob.glob(os.path.join(fdir, "*")):\n        os.remove(old)\n\n    frame_rc = None\n    frames = []\n    if with_picture:\n        fproc = subprocess.run(\n            ["ffmpeg", "-y", "-v", "error", "-i", path, "-vf", "fps=1",\n             os.path.join(fdir, "f%03d.png")], capture_output=True, text=True)\n        frame_rc = fproc.returncode\n        frames = sorted(glob.glob(os.path.join(fdir, "*.png")))\n        if frame_rc != 0:\n            raise FeedProofError(\n                f"frame extraction failed for {path}: rc={frame_rc}: "\n                + (fproc.stderr or "")[-500:])\n        if not frames:\n            raise FeedProofError(f"frame extraction produced no frames: {path}")\n\n    wav = os.path.join(fdir, "audio.wav")\n    aproc = subprocess.run(\n        ["ffmpeg", "-y", "-v", "error", "-i", path, "-vn", "-ac", "1",\n         "-ar", "16000", wav], capture_output=True, text=True)\n    audio_rc = aproc.returncode\n    if audio_rc != 0:\n        raise FeedProofError(\n            f"audio extraction failed for {path}: rc={audio_rc}: "\n            + (aproc.stderr or "")[-500:])\n    if not os.path.exists(wav):\n        raise FeedProofError(f"audio extraction produced no wav for {path}")\n    audio_np, sr = librosa.load(wav, sr=16000, mono=True)\n    audio_np = np.asarray(audio_np, dtype=np.float32)\n\n    src_dur = media_duration(path)\n    if src_dur is None:\n        raise FeedProofError(f"could not read media duration for {path}")\n    unit_count = int(round(src_dur))\n    if unit_count <= 0:\n        raise FeedProofError(f"media duration rounds to zero units: {path}")\n    if unit_count > 15:\n        unit_count = 15\n    track_dur = len(audio_np) / sr if sr else 0.0\n    if len(audio_np) == 0:\n        raise FeedProofError(f"extracted track is empty for {path}")\n    if track_dur < unit_count - 0.5:\n        raise FeedProofError(\n            f"extracted track is short for {path}: {track_dur:.3f}s of "\n            f"track for {unit_count} units")\n\n    if with_picture and len(frames) < unit_count - 1:\n        raise FeedProofError(\n            f"frame extraction is short for {path}: {len(frames)} frames "\n            f"for {unit_count} units")\n\n    per_unit_rms = []\n    per_unit_padding = []\n    zero_chunks = 0\n    for i in range(unit_count):\n        chunk = audio_np[sr * i: sr * (i + 1)]\n        if len(chunk) == 0:\n            raise FeedProofError(\n                f"unit {i} has no audio samples for {path}")\n        per_unit_rms.append(round(rms_db(chunk), 2))\n        if not np.any(chunk):\n            zero_chunks += 1\n        per_unit_padding.append(int(sr - len(chunk)) if len(chunk) < sr else 0)\n\n    overall_rms = round(rms_db(audio_np), 2)\n    nonzero_share = float(np.count_nonzero(audio_np) / len(audio_np))\n    if expected_sound and (nonzero_share == 0.0 or overall_rms <= -119.5\n                           or zero_chunks == unit_count):\n        raise FeedProofError(\n            f"track that should carry sound decoded silent for {path}: "\n            f"rms={overall_rms} dB, nonzero share={nonzero_share}")\n    if expected_silence and nonzero_share > 0.001:\n        raise FeedProofError(\n            f"declared silence track is not silent for {path}: "\n            f"nonzero share={nonzero_share}")\n\n    track_hash = hashlib.sha256(audio_np.tobytes()).hexdigest()\n    track_name = (re.sub(r"[^A-Za-z0-9]+", "_", os.path.basename(path))\n                  + "__" + track_hash[:12] + ".wav")\n    track_paths = []\n    for d in TRACK_DIRS:\n        try:\n            tp = os.path.join(d, track_name)\n            if not os.path.exists(tp):\n                sf.write(tp, audio_np, sr, subtype="PCM_16")\n            track_paths.append(tp)\n        except Exception:\n            pass\n\n    contents = []\n    for i in range(unit_count):\n        chunk = audio_np[sr * i: sr * (i + 1)]\n        if len(chunk) < sr:\n            chunk = np.pad(chunk, (0, sr - len(chunk))).astype(np.float32)\n        if with_picture:\n            img = Image.open(frames[min(i, len(frames) - 1)]).convert("RGB")\n            img.thumbnail((448, 448))\n            contents.extend(["<unit>", img, chunk])\n        else:\n            contents.append(chunk)\n\n    proof = {\n        "file": os.path.basename(path),\n        "source_file_sha256": file_sha256(path),\n        "frame_ffmpeg_rc": frame_rc,\n        "audio_ffmpeg_rc": audio_rc,\n        "track_duration_s": round(track_dur, 3),\n        "sample_rate": int(sr),\n        "sample_count": int(len(audio_np)),\n        "track_rms_db": overall_rms,\n        "track_nonzero_share": round(nonzero_share, 6),\n        "unit_count": int(unit_count),\n        "per_unit_rms_db": per_unit_rms,\n        "per_unit_padding_samples": per_unit_padding,\n        "padding_samples_total": int(sum(per_unit_padding)),\n        "zero_chunk_count": int(zero_chunks),\n        "track_sha256": track_hash,\n        "track_copy_paths": track_paths,\n        "expected_sound": bool(expected_sound),\n        "expected_silence": bool(expected_silence),\n        "feed": FEED_WITH_PICTURE if with_picture else FEED_AUDIO_ALONE,\n    }\n    feed = {"contents": contents, "proof": proof}\n    _feed_cache[key] = feed\n    return feed\n\ndef ask(job, prompt, max_new_tokens):\n    with_picture = job["arm"] != "audio_alone"\n    feed = extract_feed(os.path.join(CLIPS, job["file"]), with_picture,\n                        job["expected_sound"], job["expected_silence"])\n    proof = feed["proof"]\n    print("FEED PROOF " + json.dumps(proof, sort_keys=True))\n    contents = list(feed["contents"]) + [prompt]\n    if job["arm"] == "audio_alone":\n        sys_msg = model.get_sys_prompt(mode="default", language="en")\n        res = model.chat(msgs=[sys_msg, {"role": "user", "content": contents}],\n                         tokenizer=tokenizer, sampling=False,\n                         max_new_tokens=max_new_tokens, omni_input=False,\n                         chunk_input=True, use_tts_template=False,\n                         generate_audio=False, max_slice_nums=1,\n                         use_image_id=False)\n    else:\n        sys_msg = model.get_sys_prompt(mode="omni", language="en")\n        res = model.chat(msgs=[sys_msg, {"role": "user", "content": contents}],\n                         tokenizer=tokenizer, sampling=False,\n                         max_new_tokens=max_new_tokens, omni_input=True,\n                         use_tts_template=False, generate_audio=False,\n                         max_slice_nums=1, use_image_id=False)\n    text = answer_text(res).strip()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return text, proof\n\n# --- Test definitions ------------------------------------------------------------\nSWAP_FILES = {\n    "silence": "swap_silence.mp4",\n    "speech": "swap_speech.mp4",\n    "beeps": "swap_beeps.mp4",\n    "water": "swap_water.mp4",\n    "vacuum": "swap_vacuum.mp4",\n    "sizzle": "swap_sizzle.mp4",\n    "mixed": "swap_mixed.mp4",\n}\n\nPRESENCE_PROMPTS = {\n    "speech": "Is there a person speaking in this clip? Answer yes or no.",\n    "silence": "Is this clip silent? Answer yes or no.",\n    "water": "Is there the sound of running water in this clip? Answer yes or no.",\n    "beeps": "Is there a beeping sound in this clip? Answer yes or no.",\n}\nDETAIL_PROMPTS = {\n    "speech": "Write down any words spoken in this clip.",\n    "silence": "Is there any sound at all in this clip? Answer yes or no.",\n    "water": "What sound, if any, do you hear in this clip? Answer with a short sound name or none.",\n    "beeps": "What sound, if any, do you hear in this clip? Answer with a short sound name or none.",\n}\n\ndef base_job(test, sound, file, arm, version, expected_sound):\n    return {"test": test, "sound": sound, "file": file, "arm": arm,\n            "version": version,\n            "expected_sound": bool(expected_sound),\n            "expected_silence": not bool(expected_sound)}\n\nJOBS = []\n\n# Test 2: the 2x2. Speech baseline first, silence as the negative\n# control, then water and beeps. Arms: audio alone, then the same\n# sound with the swap still picture. Forms: open, then the two direct\n# asks (presence, and detail: transcription for speech, a short sound\n# name or the any-sound control for the others). Two repeats per cell.\nfor sound in ("speech", "silence", "water", "beeps"):\n    expected = sound != "silence"\n    for arm, version in (("audio_alone", "no_picture"),\n                         ("audio_with_still", "original_swap_still")):\n        for rep in (1, 2):\n            job = base_job("test2_2x2", sound, SWAP_FILES[sound], arm,\n                           version, expected)\n            JOBS.append((job, "open", "open_json", sound, PROMPT_JSON,\n                         rep, 300))\n            job = base_job("test2_2x2", sound, SWAP_FILES[sound], arm,\n                           version, expected)\n            JOBS.append((job, "direct", "presence", sound,\n                         PRESENCE_PROMPTS[sound], rep, 32))\n            job = base_job("test2_2x2", sound, SWAP_FILES[sound], arm,\n                           version, expected)\n            label = "transcription" if sound == "speech" else "detail"\n            JOBS.append((job, "direct", label, sound,\n                         DETAIL_PROMPTS[sound], rep, 128))\n\n# Test 3: yes/no scoring on the full swap set. Three versions per\n# sound: sound on with the still picture, muted with the still\n# picture (the true silence file, declared in the job), and sound on\n# with a plain gray frame. Matching target per file, plus the two\n# lures: the speech file is also asked about water, and the water\n# file is also asked about speech. The mixed file carries two sounds,\n# so it gets both of its matching targets.\nTARGETS = {\n    "silence": ["silence"],\n    "speech": ["a person speaking"],\n    "beeps": ["beeping"],\n    "water": ["running water"],\n    "vacuum": ["a vacuum cleaner"],\n    "sizzle": ["sizzling"],\n    "mixed": ["a person speaking", "a vacuum cleaner"],\n}\nLURES = {"speech": ["running water"], "water": ["a person speaking"]}\nfor sound in ("silence", "speech", "beeps", "water", "vacuum",\n              "sizzle", "mixed"):\n    for version in ("sound_on_still", "muted_still", "sound_on_gray"):\n        if version == "sound_on_still":\n            file = SWAP_FILES[sound]\n            expected = sound != "silence"\n        elif version == "muted_still":\n            file = SWAP_FILES["silence"]\n            expected = False\n        else:\n            file = f"series2_gray_{sound}.mp4"\n            expected = sound != "silence"\n        for target, label in ([(t, "matching_yes_no") for t in TARGETS[sound]]\n                              + [(t, "lure_yes_no")\n                                 for t in LURES.get(sound, [])]):\n            job = base_job("test3_yes_no", sound, file, "audio_with_frame",\n                           version, expected)\n            prompt = (f"Is there the sound of {target} in this clip? "\n                      "Answer yes or no.")\n            JOBS.append((job, "yes_no", label, target, prompt, 1, 16))\n\n# Test 4: tone-shift timing. Three files, one tone placed at about 2,\n# 6, and 10 seconds. Three separate asks per file: presence,\n# early-or-late, numeric. Timing is scored later, and only where\n# presence was detected; the notebook records the raw answers.\nfor start_s in (2, 6, 10):\n    file = f"series2_tone_{start_s}s.mp4"\n    for form, label, prompt, tokens in (\n            ("presence", "presence",\n             "Is there a tone in this clip? Answer yes or no.", 16),\n            ("early_late", "early_late",\n             "Does the tone start in the first half or the second half "\n             "of the clip?", 32),\n            ("numeric", "numeric_start",\n             "At what second does the tone start? Answer with a number.",\n             32)):\n        job = base_job("test4_tone_shift", "tone", file, "audio_with_still",\n                       "tone_shift_still", True)\n        job["target_start_s"] = start_s\n        JOBS.append((job, form, label, "tone", prompt, 1, tokens))\n\nprint(f"diagnostic jobs: {len(JOBS)} asks "\n      f"(test 2: 48, test 3: {sum(1 for j in JOBS if j[0][\'test\'] == \'test3_yes_no\')}, "\n      f"test 4: 9)")\n\nFIELDS = ["test", "sound", "file", "arm", "version", "question_form",\n          "question_label", "target", "prompt", "repeat",\n          "target_start_s", "expected_sound", "expected_silence",\n          "frame_ffmpeg_rc", "audio_ffmpeg_rc", "track_duration_s",\n          "sample_rate", "sample_count", "track_rms_db",\n          "track_nonzero_share", "unit_count", "per_unit_rms_db",\n          "per_unit_padding_samples", "padding_samples_total",\n          "zero_chunk_count", "track_sha256", "track_copy_paths",\n          "source_file_sha256", "raw_answer", "yes_score",\n          "yes_score_status", "model", "feed"]\n\nrows = []\nfailed = 0\n\ndef save_rows():\n    out_fields = FIELDS\n    for out_path in ("/content/drive/MyDrive/project-k-clips/series2_minicpm_results.csv",\n                     "/content/series2_minicpm_results.csv"):\n        try:\n            with open(out_path, "w", newline="") as fh:\n                w = csv.DictWriter(fh, fieldnames=out_fields)\n                w.writeheader()\n                w.writerows(rows)\n            print("saved:", out_path)\n        except Exception as e:\n            print("could not save", out_path, "-", str(e)[:120])\n\nfor job, form, label, target, prompt, rep, max_tokens in JOBS:\n    print("=" * 70)\n    print(f"{job[\'test\']} - sound={job[\'sound\']} arm={job[\'arm\']} "\n          f"version={job[\'version\']} form={form}/{label} "\n          f"target={target} repeat={rep}")\n    raw_text = ""\n    error = ""\n    proof = None\n    try:\n        raw, proof = ask(job, prompt, max_tokens)\n        if not raw.strip():\n            error = "ERROR: empty answer returned"\n            failed += 1\n            print("WARNING: this ask came back EMPTY. Recorded as an error,")\n            print("never as silence.")\n            raw_text = error\n        else:\n            raw_text = raw\n            print("RAW ANSWER:")\n            print(raw)\n    except FeedProofError as e:\n        print()\n        print("!" * 70)\n        print("FEED PROOF HARD STOP. The feed for this ask failed its proof:")\n        print(str(e))\n        print("No answer was taken for this ask, and the run stops here.")\n        print("Rows completed before this ask are saved now. Show this")\n        print("whole output to Muse.")\n        print("!" * 70)\n        save_rows()\n        sys.exit(7)\n    except Exception as e:\n        error = "ERROR: " + traceback.format_exc()\n        failed += 1\n        print(f"WARNING: this ask FAILED with an error. The full error is")\n        print("saved in the CSV column raw_answer.")\n        raw_text = error\n        proof = proof or {}\n    p = proof or {}\n    rows.append({\n        "test": job["test"], "sound": job["sound"], "file": job["file"],\n        "arm": job["arm"], "version": job["version"],\n        "question_form": form, "question_label": label, "target": target,\n        "prompt": prompt, "repeat": rep,\n        "target_start_s": job.get("target_start_s", ""),\n        "expected_sound": job["expected_sound"],\n        "expected_silence": job["expected_silence"],\n        "frame_ffmpeg_rc": p.get("frame_ffmpeg_rc", ""),\n        "audio_ffmpeg_rc": p.get("audio_ffmpeg_rc", ""),\n        "track_duration_s": p.get("track_duration_s", ""),\n        "sample_rate": p.get("sample_rate", ""),\n        "sample_count": p.get("sample_count", ""),\n        "track_rms_db": p.get("track_rms_db", ""),\n        "track_nonzero_share": p.get("track_nonzero_share", ""),\n        "unit_count": p.get("unit_count", ""),\n        "per_unit_rms_db": json.dumps(p.get("per_unit_rms_db", "")),\n        "per_unit_padding_samples": json.dumps(\n            p.get("per_unit_padding_samples", "")),\n        "padding_samples_total": p.get("padding_samples_total", ""),\n        "zero_chunk_count": p.get("zero_chunk_count", ""),\n        "track_sha256": p.get("track_sha256", ""),\n        "track_copy_paths": json.dumps(p.get("track_copy_paths", "")),\n        "source_file_sha256": p.get("source_file_sha256", ""),\n        "raw_answer": raw_text,\n        "yes_score": "not exposed",\n        "yes_score_status": ("model.chat returns decoded text only and "\n                             "does not request output scores"),\n        "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                  "[venv py3.12, torch 2.8.0+cu126]"),\n        "feed": p.get("feed", ""),\n    })\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nif failed:\n    print()\n    print("!" * 70)\n    print(f"RUN PROBLEM: {failed} ask(s) failed or came back empty. Their")\n    print("raw words or full errors are in the CSV column raw_answer.")\n    print("This run is NOT clean evidence until those asks are explained.")\n    print("!" * 70)\nelse:\n    print()\n    print("Clean run: every ask answered, every feed proved in print first.")\n\nsave_rows()\nif failed:\n    sys.exit(6)\n'
with open("/content/minicpm_series2_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/minicpm_series2_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, ladder, sanity -------------------------------------------
import os, subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_series2_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a ladder rung, or the sanity question.")
    print("Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, sanity answered.")
print("Go on to Step 4.")

## Step 4 · Run the diagnostic battery

The next cell runs the runner in **battery mode**: proof and ladder
again, sanity again, then the three tests. Every ask prints its
FEED PROOF line before the model's answer is taken. Watch for it:
that line is the whole point of Series Two.

**Test 2, the 2x2 (48 asks).** Sounds in a fixed order: speech
first (the baseline), then true silence (the negative control),
then water, then beeps. Each sound is fed two ways: as sound alone,
and as the same soundtrack with the swap still picture. The
sound-alone arm uses the model's unified `model.chat` with no
picture: the code has no separate audio-only method, and the
notebook states that difference rather than hiding it. Each arm is
asked three ways, two repeats each: the open repaired JSON question
(word for word the Series One question), a direct presence question
made for that sound, and a direct detail question (for speech:
"Write down any words spoken in this clip."; for the others, a
short sound-name question, and for silence an any-sound control).

**Test 3, yes/no scoring (30 asks).** All seven swap soundtracks,
three versions each: sound on with the still picture, muted with
the still picture, and sound on with a plain gray frame. Each
version is asked about its matching sound ("Is there the sound of
X in this clip? Answer yes or no."), the mixed file about both of
its sounds, and two lures ride along: the speech file is asked
about water, and the water file is asked about speech. The reading
is fixed in advance in the Series Two plan: yes not moving between
sound on and muted means the sound never got in; yes rising while
the free answer still says none means the loss is at reporting;
yes staying high on a muted kitchen picture means the scene is
guessing. Scores are not exposed by `model.chat`, so the yes-score
column reads "not exposed" on every row and the words are the
record.

**Test 4, tone-shift timing (9 asks).** The three tone files from
Step 2. Per file, three separate asks: presence (yes or no),
early-or-late ("Does the tone start in the first half or the
second half of the clip?"), and numeric ("At what second does the
tone start? Answer with a number."). Timing gets scored later,
only on files where presence was detected, so a timing failure
can no longer hide inside a hearing failure.

**Hard stops and warnings.** A feed that fails its proof stops the
whole run on the spot (exit code 7), with the rows completed so
far saved first. An ask that errors or comes back empty is saved
with its full error text in `raw_answer` and a WARNING on screen,
never as silence, and the run ends with the RUN PROBLEM banner
(exit code 6). Results save to `series2_minicpm_results.csv` in
the Drive clips folder, one row per ask, with the full feed-proof
numbers on every row.

In [ ]:
# --- Battery mode: the Series Two diagnostic tests --------------------------------
import os, subprocess
VENV_PY = "/content/minicpm-venv/bin/python"
proc = subprocess.run([VENV_PY, "/content/minicpm_series2_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: a gate failed, a feed proof hard-stopped the run,")
    print("or some asks are broken and marked in the CSV. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: every ask answered, every feed proved, CSV saved.")

## Closing · How to read the answers

- **Feed proof first.** A row is only as good as its proof line.
  Rows after a hard stop do not exist, on purpose: a stopped run is
  a named failure, not a pile of quiet none answers.
- **Test 2 reads as a square.** Sound alone against sound with a
  picture separates the picture's interference; open against direct
  separates the question's suppression. Speech is the baseline and
  silence is the control: a yes on silence is a false alarm, and
  the rate matters as much as the hits.
- **Test 3 reads by movement.** Compare each sound across its
  three versions before comparing sounds to each other. The
  pattern of what moves, and what does not, is the result.
- **Test 4 reads in order.** Presence first. Early-or-late and
  numeric starts count only where presence was a hit.
- **Scope.** Every claim stays capped at this model, this feed,
  these files, this date. The stages in the Series Two plan stay
  hypotheses until an intervention separates them, and these three
  tests are the interventions.